In [ ]:
!pip install agentpy ipywidgets matplotlib seaborn
!jupyter labextension install @jupyter-widgets/jupyterlab-manager

In [ ]:
import agentpy as ap
import numpy as np
import random

# Agente: Carro
class DummyCar(ap.Agent):
    def setup(self):
        self.waiting = False  # Si está esperando en un semáforo rojo

    def step(self):
        env = self.model.environment
        x, y = env.positions[self]
        next_pos = env.next_position(x, y)
        traffic_light = env.traffic_light_at(next_pos)
        # Verifica si la siguiente posición está ocupada por otro carro
        occupied = any(pos == next_pos for car, pos in env.positions.items() if car in env.cars and car != self)
        # Se detiene si hay semáforo en rojo o si la posición está ocupada
        if (traffic_light and not traffic_light.is_green) or occupied:
            self.waiting = True
        else:
            self.waiting = False
            env.move_car(self, next_pos)

# Agente: Semáforo
class DummyTrafficLight(ap.Agent):
    def setup(self):
        self.is_green = random.choice([True, False])
        self.timer = 0

    def step(self):
        self.timer += 1
        # Cambia cada 5 pasos
        if self.timer % 5 == 0:
            self.is_green = not self.is_green

# Ambiente: Rotonda más grande
class RoundaboutEnvironment(ap.Grid):
    def setup(self):
        self.cars = self.model.cars
        self.traffic_lights = self.model.traffic_lights
        self.road_positions = [
            (1,3), (1,4), (2,5), (3,5), (4,5), (5,4), (5,3), (5,2),
            (4,1), (3,1), (2,1), (1,2)
        ]
        # Posiciones fijas para semáforos (ajustadas para 7x7)
        self.traffic_light_positions = [(1,3), (3,5), (5,3), (3,1)]
        for tl, pos in zip(self.traffic_lights, self.traffic_light_positions):
            self.add_agents([tl], positions=[pos])
        # Posiciones iniciales para carros (ajustadas para 7x7)
        for car, pos in zip(self.cars, self.road_positions):
            self.add_agents([car], positions=[pos])

    def next_position(self, x, y):
        idx = self.road_positions.index((x, y))
        return self.road_positions[(idx + 1) % len(self.road_positions)]
        
    def traffic_light_at(self, pos):
        for tl in self.traffic_lights:
            if self.positions[tl] == pos:
                return tl
        return None
    def move_car(self, car, new_pos):
        self.move_to(car, new_pos)

# Modelo
class RoundaboutModel(ap.Model):
    def setup(self):
        n_cars = self.p.n_cars
        n_lights = self.p.n_lights
        self.cars = ap.AgentList(self, n_cars, DummyCar)
        self.traffic_lights = ap.AgentList(self, n_lights, DummyTrafficLight)
        self.environment = RoundaboutEnvironment(self, (7,7))
        self.environment.setup()

    def step(self):
        self.traffic_lights.step()
        self.cars.step()

# Parámetros
parameters = {
    'n_cars': 4,
    'n_lights': 4,
    'steps': 20
}

model = RoundaboutModel(parameters)
results = model.run()

# Visualización (puedes adaptar la función de Act1 para mostrar el grid)

In [ ]:
from IPython.display import HTML
import matplotlib.pyplot as plt
import seaborn as sns

def roundabout_plot(model, ax):
    grid = np.zeros(model.environment.shape)
    # 0 = vacío, 1 = semáforo rojo, 2 = semáforo verde, 3 = carro
    for tl in model.traffic_lights:
        pos = model.environment.positions[tl]
        grid[pos] = 2 if tl.is_green else 1
    for car in model.cars:
        pos = model.environment.positions[car]
        grid[pos] = 3
    ax.clear()
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(['#ffffff', '#d9534f', '#5cb85c', '#000000'])
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=3)
    ax.set_title('Rotonda: Negro=Carro, Rojo=Semáforo Rojo, Verde=Semáforo Verde')
    ax.set_xlabel('Columnas')
    ax.set_ylabel('Filas')
    ax.set_xticks([])
    ax.set_yticks([])

model = RoundaboutModel(parameters)
fig, ax = plt.subplots()
animation = ap.animate(model, fig, ax, roundabout_plot)
HTML(animation.to_jshtml())